# Limpeza do CAD da planta → geometria do CFD (LD Celulose)
Demonstração do caminho completo, em 5 etapas:

1. **Juntar as partes do RAR** e extrair o STEP
2. **Ler o STEP com os nomes** das peças (OpenCascade/XCAF)
3. **Inventário**: nome + caixa envolvente de cada peça → CSV
4. **Classificar** por nome (nomenclatura de projeto), cota e tag de equipamento
5. **Reconstruir** cada equipamento como cilindro ou caixa → STEP limpo para o STAR-CCM+

Critérios: `05_limpeza_CAD_criterios.md` · método: `06_metodologia_geometria_CFD.md`.

**Rodar em ordem.** Para uma demonstração rápida, use um subconjunto pequeno (ex.: o civil `#F-C-1219`, 15 MB).

> **Demonstração × geometria final:** este notebook é a versão automática. Na geometria final (38 sólidos) houve revisão de engenharia: o prédio de O₃ foi dividido (corpo baixo, bloco alto, terraço coberto, sala de transformadores) a partir das paredes, as descargas dos exaustores vieram da geometria real dos bocais, e equipamentos pequenos foram descartados. Exemplo: no civil `#F-C-1219` o automático devolve **uma** envoltória de 31 × 30 × 21 m; o modelo final usa 5 blocos.

In [ ]:
# 0) Instalação — versões fixas (a cadquery-ocp 8.0 mudou a API)  ~1 min
!pip install -q "cadquery==2.8.0" "cadquery-ocp==7.9.3.1.1"
!apt-get -qq install -y unrar > /dev/null 2>&1 || true
# Se pedir para reiniciar a sessão depois da instalação: Ambiente de execução → Reiniciar sessão, e siga da próxima célula.

## Parâmetros
`ORIGEM` = ponto de referência em coordenadas da planta [m] (canto do prédio de O₃ e cota do piso). As distâncias e zonas são medidas a partir dele.

In [ ]:
ORIGEM  = (20937.39, 9401.4, 970.4)   # x, y, z [m] — canto SW do prédio de O₃ e cota do piso
RAIO_Z0 = 65.0    # zona de detalhe (fonte) [m] → remove o que for < 1 m
RAIO_Z1 = 300.0   # zona de interesse [m]        → remove o que for < 3 m
SAIDA   = '/content/resultado'
OX, OY, OZ = ORIGEM
import os; os.makedirs(SAIDA, exist_ok=True)

## Enviar o(s) arquivo(s)
Selecione **todas as partes** do RAR (`.part001.rar`, `.part002.rar`, …) de uma vez — ou um `.stp` direto.

In [ ]:
from google.colab import files
enviados = files.upload()
nomes = sorted(enviados.keys())
import re
ENTRADA = '/content/' + (next((n for n in nomes if re.search(r'part0*1\.rar$', n, re.I)), None) or nomes[0])
print('entrada:', ENTRADA)

In [ ]:
# Bibliotecas
import csv, math, os, re, subprocess, time, glob
from collections import defaultdict
import cadquery as cq
from OCP.STEPCAFControl import STEPCAFControl_Reader
from OCP.TDocStd import TDocStd_Document
from OCP.TCollection import TCollection_ExtendedString
from OCP.XCAFDoc import XCAFDoc_DocumentTool
from OCP.TDF import TDF_LabelSequence, TDF_Label
from OCP.TDataStd import TDataStd_Name
from OCP.TopLoc import TopLoc_Location
from OCP.Bnd import Bnd_Box
from OCP.BRepBndLib import BRepBndLib
from OCP.IFSelect import IFSelect_RetDone


## Etapa 1 — juntar as partes do RAR e extrair o STEP

In [ ]:
# ── ETAPA 1 ── juntar as partes do RAR (o unrar acha .part002, .part003… sozinho) ─────────────────
step = ENTRADA
if step.lower().endswith(".rar"):
    subprocess.run(["unrar", "e", "-o+", "-inul", step, SAIDA + os.sep], check=True)
    step = max((os.path.join(SAIDA, f) for f in os.listdir(SAIDA) if f.lower().endswith((".stp", ".step"))),
               key=os.path.getsize)
print(f"[1] STEP: {step} ({os.path.getsize(step)/1e6:.0f} MB)")


## Etapa 2 — ler o STEP **com os nomes**
O leitor simples de STEP perde os nomes; o XCAF mantém a árvore da montagem. Para arquivos de 500 MB leva ~5 min.

In [ ]:
# ── ETAPA 2 ── ler COM NOMES (o leitor simples do STEP perde os nomes; o XCAF mantém a árvore) ───
t0 = time.time()
doc = TDocStd_Document(TCollection_ExtendedString("doc"))
leitor = STEPCAFControl_Reader(); leitor.SetNameMode(True)
assert leitor.ReadFile(step) == IFSelect_RetDone, "falha ao ler o STEP"
leitor.Transfer(doc)
arvore = XCAFDoc_DocumentTool.ShapeTool_s(doc.Main())
print(f"[2] lido com nomes em {time.time()-t0:.0f} s")

def nome(lab):
    n = TDataStd_Name()
    return n.Get().ToExtString() if lab.FindAttribute(TDataStd_Name.GetID_s(), n) else "?"


## Etapa 3 — inventário (nome + caixa envolvente de cada peça)

In [ ]:
# ── ETAPA 3 ── inventário: percorre a árvore, aplica as transformações e mede cada peça ────────────
pecas = []   # (caminho, nome, [xmin, ymin, zmin, xmax, ymax, zmax] em metros e coordenadas LOCAIS)
def visita(lab, loc, caminho):
    if arvore.IsReference_s(lab):                       # ocorrência → segue para a peça, somando a posição
        ref = TDF_Label(); arvore.GetReferredShape_s(lab, ref)
        visita(ref, loc.Multiplied(arvore.GetLocation_s(lab)), caminho + [nome(lab)]); return
    if arvore.IsAssembly_s(lab):                         # subconjunto → visita os filhos
        filhos = TDF_LabelSequence(); arvore.GetComponents_s(lab, filhos, False)
        for i in range(1, filhos.Length() + 1): visita(filhos.Value(i), loc, caminho)
        return
    caixa = Bnd_Box(); BRepBndLib.Add_s(arvore.GetShape_s(lab).Moved(loc), caixa, False)
    if caixa.IsVoid(): return
    x0, y0, z0, x1, y1, z1 = (v / 1000 for v in caixa.Get())          # mm → m
    if z0 < 100: return                                               # peça sem posição (na origem do arquivo)
    pecas.append((" / ".join(caminho[-3:]), caminho[-1] if caminho else nome(lab),
                  [x0 - OX, y0 - OY, z0 - OZ, x1 - OX, y1 - OY, z1 - OZ]))
raizes = TDF_LabelSequence(); arvore.GetFreeShapes(raizes)
for i in range(1, raizes.Length() + 1): visita(raizes.Value(i), TopLoc_Location(), [])
with open(os.path.join(SAIDA, "inventario.csv"), "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f); w.writerow(["caminho", "peca", "xmin", "ymin", "zmin", "xmax", "ymax", "zmax"])
    for c, n, b in pecas: w.writerow([c, n] + [round(v, 3) for v in b])
print(f"[3] inventário: {len(pecas)} peças → inventario.csv")

import pandas as pd
pd.read_csv(os.path.join(SAIDA,'inventario.csv')).head(15)

## Etapa 4 — classificar por nome, cota e tag
As regras são a lista `REGRAS_SAI`: edite e rode de novo para ver o efeito.

In [ ]:
# ── ETAPA 4 ── classificar: NOME (nomenclatura de projeto) + TAMANHO (malha) + DISTÂNCIA (zona) ────
REGRAS_SAI = [  # (padrão no nome, motivo) — critérios do 05_limpeza_CAD_criterios.md §4
    (r"PILE|BLOCK_P\d|BASE|FUND|TUB-AP|P\.R-",          "fundação / abaixo do piso"),
    (r"CRANE|GUINDAST|SF100FT|TRUCK|CAMINH",             "temporário / veículo"),
    (r"FITTING|SCTN|FRMWORK|SBFRAMEWORK|BEANS|COLUMN",   "estrutura / perfil (treliça)"),
    (r"ESCADA|STAIR|HANDRAIL|GUARDA|PANEL|HANGER",       "acessório"),
    (r"^PS-|AIDGROUP|INSERT|_INS-|NOZZ|FLANGE|VALV",     "suporte / aterramento / bocal / válvula"),
]
TAG = re.compile(r"(1[0-9]{3}-\d{2}-\d{3})")            # tag de equipamento AVEVA: área-classe-número

def distancia(b):  # distância horizontal da caixa até a origem (fonte)
    return math.hypot(max(0, b[0], -b[3]), max(0, b[1], -b[4]))

grupos, saiu = defaultdict(list), defaultdict(int)
for caminho, n, b in pecas:
    texto = caminho + " " + n
    if re.search(r"^(GWALL|STWALL)", n) and b[5] >= 0.5:   # paredes de prédio (sem tag) → envoltória do prédio
        grupos["envoltoria_" + caminho.split(" / ")[0][:20]].append(b); continue
    motivo = next((m for p, m in REGRAS_SAI if re.search(p, n, re.I)), None)
    tag = TAG.search(texto)
    if b[5] < 0.5:                       motivo = "abaixo do piso"
    elif tag and motivo in (None, "suporte / aterramento / bocal / válvula", "acessório"):
        motivo = None                    # peça de equipamento (bocal, escada do vaso…): decide o conjunto (etapa 5)
    if motivo: saiu[motivo] += 1; continue
    grupos[tag.group(1) if tag else "sem tag"].append(b)
print("[4] saíram:", dict(saiu))


## Etapa 5 — um objeto por tag: cilindro (planta ~quadrada) ou caixa → STEP limpo

In [ ]:
# ── ETAPA 5 ── um objeto por TAG: caixa envolvente do conjunto → cilindro (planta ~quadrada) ou caixa ─
asm, tabela = cq.Assembly(), []
for tag, caixas in grupos.items():
    if tag == "sem tag": continue
    b = [min(c[0] for c in caixas), min(c[1] for c in caixas), max(0, min(c[2] for c in caixas)),
         max(c[3] for c in caixas), max(c[4] for c in caixas), max(c[5] for c in caixas)]
    dx, dy, h = b[3] - b[0], b[4] - b[1], b[5] - b[2]
    d = distancia(b)
    limite = 1.0 if d <= RAIO_Z0 else 3.0 if d <= RAIO_Z1 else 1e9   # critério de tamanho por zona (§3)
    if max(dx, dy, h) < limite or h < 1.0: continue                         # pequeno, ou base/pedestal baixo
    if abs(dx - dy) / max(dx, dy) < 0.25 and not tag.startswith("envoltoria"):  # planta ~quadrada → vaso/silo/torre
        s = cq.Workplane("XY").circle(min(dx, dy) / 2).extrude(h).translate(((b[0]+b[3])/2, (b[1]+b[4])/2, b[2]))
        forma = "cilindro"
    else:
        s = cq.Workplane("XY").box(dx, dy, h, centered=False).translate((b[0], b[1], b[2]))
        forma = "caixa"
    asm.add(s.val().scale(1000), name=tag);   # STEP do CadQuery é gravado em mm
    tabela.append([tag, forma, round(d), round(dx, 1), round(dy, 1), round(h, 1)])
if tabela: asm.export(os.path.join(SAIDA, "geometria_limpa.step"))
print(f"[5] {len(tabela)} objetos → geometria_limpa.step")
for t in sorted(tabela, key=lambda t: t[2]): print(f"    {t[0]:14s} {t[1]:8s} dist {t[2]:4d} m   {t[3]} × {t[4]} m, h {t[5]} m")


## Visualização — antes × depois (vista de cima)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))
for _, _, b in pecas:
    ax1.add_patch(Rectangle((b[0], b[1]), b[3]-b[0], b[4]-b[1], fill=False, lw=.3, ec='gray'))
ax1.set_title(f'ANTES — {len(pecas)} peças')
for t in tabela:
    tag = t[0]; cx = grupos[tag]
    x0, y0 = min(c[0] for c in cx), min(c[1] for c in cx); x1, y1 = max(c[3] for c in cx), max(c[4] for c in cx)
    ax2.add_patch(Rectangle((x0, y0), x1-x0, y1-y0, fc='#4C78A8', alpha=.6)); ax2.text((x0+x1)/2, (y0+y1)/2, tag[-6:], fontsize=6, ha='center')
ax2.set_title(f'DEPOIS — {len(tabela)} objetos')
for ax in (ax1, ax2):
    ax.plot(0, 0, 'r*', ms=14); ax.set_aspect('equal'); ax.autoscale(); ax.grid(alpha=.3); ax.set_xlabel('x [m]'); ax.set_ylabel('y [m]')
plt.show()

## Baixar os resultados

In [ ]:
files.download(os.path.join(SAIDA, 'inventario.csv'))
files.download(os.path.join(SAIDA, 'geometria_limpa.step'))